# Ensemble Of Catboost, XGBoost & LightGBM

## Bulk Imports Modules

In [ ]:
# import required modules and libraries
import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, average_precision_score, classification_report, confusion_matrix,
    precision_recall_curve
)
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier as cbc
from xgboost import XGBClassifier as xgbc
from lightgbm import LGBMClassifier as lgbmc

## Data Input

In [ ]:
# Set paths
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e9'
TRAIN_PATH = os.path.join(DATA_DIR, 'train.csv')
TEST_PATH = os.path.join(DATA_DIR, 'test.csv')
SUBMISSION_PATH = os.path.join(DATA_DIR, 'sample_submission.csv')

# Load data
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBMISSION_PATH)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

## Data Processing

In [ ]:
# Data Preprocessing

# Identify categorical and numerical columns
# using test so that we don't need to drop target feature from the categorical_cols ahead
categorical_cols = test.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = test.select_dtypes(include=['int64', 'float64']).columns.tolist()

# convert default categorical features of dataframe to light-gbm native categorical handling dtype
for cat_feature in categorical_cols:
    train[cat_feature] = train[cat_feature].astype("category")
    test[cat_feature] = test[cat_feature].astype("category")

# Target variable mapping
target_col = 'Will_Buy_EV'

if train[target_col].dtype == 'category':
    train[target_col] = train[target_col].map({'No': 0, 'Yes': 1})
    print("Mapping Done...\n")

# Separate features and target
X = train.drop(columns=['id', target_col])
y = train[target_col]
X_test = test.drop(columns=['id'])

# mapping target features
y = y.map({"No":0, "Yes":1})

print("Categorical features:", categorical_cols)
print("\nNumerical features:", numerical_cols)

## Best Params

In [ ]:
# best params
best_xgb_params = {
    "n_estimators": 1430,
    "max_depth": 3,
    "learning_rate": 0.06794357538228223,
    "min_child_weight": 7,
    "subsample": 0.714084559699548,
    "colsample_bytree": 0.821710612677985,
    "gamma": 0.7479899084089237,
    "reg_alpha": 0.027771184989600878,
    "reg_lambda": 1.0357715012993982,
    "scale_pos_weight": 1.5439264692049892,
    "early_stopping_rounds": 478
}

best_catboost_params = {
    "iterations": 2827,
    "depth": 3,
    "learning_rate": 0.07828581833627082,
    "l2_leaf_reg": 6.102353961854384,
    "random_strength": 1.3138572454267994,
    "bagging_temperature": 0.039193432743536344,
    "border_count": 252,
    "early_stopping_rounds": 312
}

best_lightgbm_params = {
    "max_depth": 3,
    "n_estimators": 2118,
    "learning_rate": 0.06447836392215553,
    "num_leaves": 6,
    "min_child_samples": 81,
    "subsample": 0.8156064390376033,
    "colsample_bytree": 0.8210130019038757,
    "reg_alpha": 1.3978327472570622e-05,
    "reg_lambda": 0.17099007769746272,
    "min_split_gain": 0.16970946366885054,
    "scale_pos_weight": 2.407193478675026
}

## 5 Fold CV

In [ ]:
# 5 Fold CV
print("Starting 5-Fold Stratified Cross-Validation...")

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# OOF predictions
cbc_oof_preds = np.zeros(len(X))
xgb_oof_preds = np.zeros(len(X))
lgbm_oof_preds = np.zeros(len(X))

# Test predictions
cbc_oof_test_preds = np.zeros(len(X_test))
xgb_oof_test_preds = np.zeros(len(X_test))
lgbm_oof_test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\nTraining Fold {fold + 1}/5...")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # Create models
    cbc_model = cbc(
        loss_function="Logloss",
        **best_catboost_params,
        random_seed=42,
        verbose=200,
        task_type="GPU"
    )

    lgbm_model = lgbmc(
        objective="binary",
        **best_lightgbm_params,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    xgb_model = xgbc(
        **best_xgb_params,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        device="cuda",
        random_state=42,
        enable_categorical=True
    )

    # Train models
    cbc_model.fit(
        X_train,
        y_train,
        cat_features=categorical_cols
    )

    lgbm_model.fit(
        X_train,
        y_train,
        categorical_feature=categorical_cols
    )

    xgb_model.fit(
        X_train,
        y_train,
        eval_set=[(X_val,y_val)],
        verbose=False
    )

    # Validation predictions
    cbc_val_preds = cbc_model.predict_proba(X_val)[:, 1]
    cbc_oof_preds[val_idx] = cbc_val_preds

    xgb_val_preds = xgb_model.predict_proba(X_val)[:, 1]
    xgb_oof_preds[val_idx] = xgb_val_preds

    lgbm_val_preds = lgbm_model.predict_proba(X_val)[:, 1]
    lgbm_oof_preds[val_idx] = lgbm_val_preds

    # Test predictions
    cbc_test_preds = cbc_model.predict_proba(X_test)[:, 1]
    cbc_oof_test_preds += cbc_test_preds / 5

    xgb_test_preds = xgb_model.predict_proba(X_test)[:, 1]
    xgb_oof_test_preds += xgb_test_preds / 5

    lgbm_test_preds = lgbm_model.predict_proba(X_test)[:, 1]
    lgbm_oof_test_preds += lgbm_test_preds / 5

    # Fold ROC-AUC
    cbc_fold_auc = roc_auc_score(
        y_val,
        cbc_val_preds
    )

    xgb_fold_auc = roc_auc_score(
        y_val,
        xgb_val_preds
    )

    lgbm_fold_auc = roc_auc_score(
        y_val,
        lgbm_val_preds
    )    

    print(f"Fold {fold + 1} Catboost ROC-AUC: {cbc_fold_auc:.4f}")
    print(f"Fold {fold + 1} XGBoost ROC-AUC: {xgb_fold_auc:.4f}")
    print(f"Fold {fold + 1} LightGBm ROC-AUC: {lgbm_fold_auc:.4f}")
    print()


print("\nCross-Validation complete!")

## Optimized Weights Via Grid Search

In [ ]:
# Grid search to find best weights for ensembling
best_auc = 0
best_weights = None

for w_cbc in np.arange(0, 1.01, 0.05):
    for w_xgb in np.arange(0, 1.01 - w_cbc, 0.05):
        w_lgbm = 1.0 - w_cbc - w_xgb

        blend = (
            w_cbc * cbc_oof_preds +
            w_xgb * xgb_oof_preds +
            w_lgbm * lgbm_oof_preds
        )

        auc = roc_auc_score(y, blend)

        if auc > best_auc:
            best_auc = auc
            best_weights = (w_cbc, w_xgb, w_lgbm)

print("Best OOF AUC:", best_auc)
print("Best weights:", best_weights)

In [ ]:
# Calculating final OOF Predictions
oof_preds = (
    best_weights[0] * cbc_oof_preds +
    best_weights[1] * xgb_oof_preds +
    best_weights[2] * lgbm_oof_preds
)
oof_test_preds = (
    best_weights[0] * cbc_oof_test_preds +
    best_weights[1] * xgb_oof_test_preds +
    best_weights[2] * lgbm_oof_test_preds
)

In [ ]:
# Overall OOF Metrics

# 0.5 threshold for classification metrics
y_pred_binary = (oof_preds >= 0.5).astype(int)

accuracy = accuracy_score(
    y,
    y_pred_binary
)

precision = precision_score(
    y,
    y_pred_binary
)

recall = recall_score(
    y,
    y_pred_binary
)

f1 = f1_score(
    y,
    y_pred_binary
)

roc_auc = roc_auc_score(
    y,
    oof_preds
)

pr_auc = average_precision_score(
    y,
    oof_preds
)


# Print Overall Metrics
print("\n" + "-" * 40)
print("CatBoost (5-Fold OOF) Performance:")
print("-" * 40)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")

print("-" * 40)

# Classification Report
print("\nClassification Report:")

print(
    classification_report(
        y,
        y_pred_binary
    )
)

# Confusion Matrix
print("Confusion Matrix:")

print(
    confusion_matrix(
        y,
        y_pred_binary
    )
)

## Precision Recall Curve

In [ ]:
# Precision Recall Curve
precision_curve, recall_curve, thresholds = precision_recall_curve(y, oof_preds)

plt.plot(
    recall_curve,
    precision_curve,
    label=f"Ensemble (PR-AUC = {pr_auc:.4f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid()
plt.show()

## Model Diversity Analysis

In [ ]:
# analyzing prediction diversity
pred_matrix = np.column_stack([
    cbc_oof_preds,
    xgb_oof_preds,
    lgbm_oof_preds
])

corr = np.corrcoef(pred_matrix, rowvar=False)

corr_df = pd.DataFrame(
    corr,
    index=["CatBoost", "XGBoost", "LightGBM"],
    columns=["CatBoost", "XGBoost", "LightGBM"]
)

print(corr_df)

## Submission

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    target_col: oof_test_preds
})

submission.to_csv(
    "submission.csv",
    index=False
)

print("Submission saved.")